# Ścieżka B · Domenowy profil ryzyka DOSPERT

**Projektowanie i badanie informatycznych systemów wspomagania decyzji** · notatnik projektu z aplikacją · szkielet do uzupełnienia

Notatnik prowadzi cały projekt ścieżki B w jednym miejscu, zgodnie z przewodnikiem *B — DOSPERT — Przewodnik*:

| Etap | Kto robi | Pliki |
|---|---|---|
| 1–4 · eksport, 30 pozycji, średnie domen, kara c | zespół (luki `# TODO`) | `qualtrics_raw.csv` → `responses_clean.csv` → `profil.csv` |
| 5 · reguła i wybór 12 par z gotowej puli | zespół (luki `# TODO`) | `model.py`, `pool.csv`, `cases.csv` |
| 6 · zamrożenie, kontrakt, aplikacja | gotowe | `session_config.json`, `app.py` |
| 7 · analiza walidacji i wrażliwości | zespół | wynik do raportu |

**Aplikacja jest gotowa.** Nie piszecie interfejsu. Aplikacja czyta pliki tworzone przez ten notatnik i działa tylko wtedy, gdy spełniają one kontrakt.

**Dwa tryby pracy** (komórka konfiguracji poniżej):
- `TRYB = 'przyklad'`: wymyślony uczestnik P01 z przewodnika. Testy sprawdzają liczby kontrolne z przewodnika. Zacznij tutaj.
- `TRYB = 'wlasne'`: eksport Waszego uczestnika. Testy funkcji działają dalej, a testy liczb P01 są pomijane: notatnik wypisuje wtedy Wasz wynik.

Notatnik używa **gotowego wariantu**, czyli puli 16 scenariuszy rekreacyjnych. Jeśli zespół przygotuje własne scenariusze (8–12 par, uzgodnione z prowadzącymi), wystarczy podmienić tabelę puli w sekcji 5. Reszta notatnika i aplikacja działają bez zmian.

> **Jak pracować.** Uruchamiaj komórki po kolei. W miejscach `# TODO` uzupełnij kod i usuń `raise NotImplementedError`. Test pod każdym krokiem pokazuje ✓ albo zatrzymuje się na ✗ z opisem. Nie zmieniaj liczb w testach.

## 0 · Konfiguracja i folder projektu na Dysku Google
W Colabie pliki znikają po rozłączeniu, dlatego folder roboczy leży na Dysku Google. Przy pierwszym uruchomieniu Colab poprosi o zgodę na dostęp do Dysku. Folder można udostępnić zespołowi i prowadzącym. Dla przykładu i dla każdego uczestnika powstaje osobny folder.

W plikach są tylko kod uczestnika i jego odpowiedzi. Informacja przed zgodą powinna mówić, gdzie przechowujecie dane i kto ma do nich dostęp.

In [ ]:
TRYB = 'przyklad'          # 'przyklad' albo 'wlasne'
PARTICIPANT = 'P01'        # kod uczestnika (taki sam w Qualtricsie i w sesji)
SESSION_ID = 'S01'         # identyfikator sesji walidacyjnej
INSTRUCTOR_PIN = '2468'    # kod widoku prowadzącego w aplikacji; nie pokazuj go uczestnikowi
UZYJ_DYSKU = True          # w Colabie: folder projektu na Dysku Google

import os, sys, hashlib, inspect, json, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if TRYB not in ('przyklad', 'wlasne'):
    raise ValueError("TRYB musi mieć wartość 'przyklad' albo 'wlasne'")
if TRYB == 'przyklad':
    PARTICIPANT = 'P01'
if IN_COLAB and UZYJ_DYSKU:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/projekt_B'
else:
    ROOT = os.path.abspath('projekt_B')
WORKDIR = os.path.join(ROOT, 'przyklad_P01' if TRYB == 'przyklad' else PARTICIPANT)
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

def check(label, cond):
    """Test ogólny: działa w obu trybach."""
    if not bool(cond):
        raise AssertionError('✗ ' + label)
    print('✓', label)

def check_example(label, cond, value=None):
    """Test liczb przykładu P01: w trybie 'wlasne' tylko wypisuje Wasz wynik."""
    if TRYB == 'przyklad':
        check(label, cond)
    else:
        print('ℹ pominięto test przykładu:', label, '' if value is None else f'| Wasz wynik: {value}')

def close(a, b, tol=1e-6):
    return a is not None and b is not None and abs(a - b) <= tol

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def file_sha256(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

def write_qualtrics_csv(path, rows, descriptions):
    """Plik w układzie eksportu Qualtricsa: 3 wiersze nagłówkowe + odpowiedzi."""
    cols = list(rows[0].keys())
    df = pd.DataFrame([[descriptions.get(c, c) for c in cols], [json.dumps({'ImportId': f'QID_{c}'}) for c in cols]]
                      + [[r.get(c, '') for c in cols] for r in rows], columns=cols)
    df.to_csv(path, index=False)

print(f'Tryb: {TRYB} | uczestnik: {PARTICIPANT} | folder: {WORKDIR}')

## 1 · Klucz pozycji i stałe (rozdz. 5 i 8)
DOSPERT 2006: 30 pozycji na skali 1–7, moduły W (gotowość), R (postrzegane ryzyko) i opcjonalny B (korzyści). Domenę wyznacza identyfikator pozycji. Mapowanie W → c jest **regułą prototypu**, a nie wynikiem DOSPERT.

In [ ]:
PROTOCOL = 'B_2006_v1'
MODEL_VERSION = 'B_linear_v1'
KEY = {'ethical': [6, 9, 10, 16, 29, 30], 'financial': [3, 4, 8, 12, 14, 18], 'health_safety': [5, 15, 17, 20, 23, 26],
       'recreational': [2, 11, 13, 19, 24, 25], 'social': [1, 7, 21, 22, 27, 28]}
DOMAIN_PL = {'ethical': 'etyczna', 'financial': 'finansowa', 'health_safety': 'zdrowie i bezpieczeństwo',
             'recreational': 'rekreacyjna', 'social': 'społeczna'}
APP_DOMAIN = 'recreational'          # gotowa pula dotyczy tylko tej domeny
C0 = 0.8                             # baseline: c0 = 0,8 odpowiada W = 2,2 (reguła ostrożna, nie neutralna)
C0_EXTRA = 0.5                       # analiza dodatkowa zapisana przed testem (W = 4)
TIE_EPS = 0.005                      # |Score lewej − Score prawej| < 0,005 → TIE
NEAR = 0.02                          # odrzucenie par z c* bliskim c albo c0
C_P01 = 13 / 36                      # c przykładu P01: stała do testów funkcji
check('klucz obejmuje 30 różnych pozycji', sorted(sum(KEY.values(), [])) == list(range(1, 31)))

## 2 · Eksport z Qualtricsa (rozdz. 5 i 7)
Pola `B_W01`–`B_W30`, `B_R01`–`B_R30` i opcjonalnie `B_B01`–`B_B30` mają odpowiedzi 1–7. W trybie `przyklad` są to wartości z przewodnika, w których rekreacja sumuje się do W = 29.

In [ ]:
ITEMS_P01 = {1: (6, 3, 5), 2: (4, 4, 5), 3: (2, 6, 4), 4: (3, 6, 4), 5: (2, 6, 2), 6: (1, 6, 2), 7: (6, 2, 5), 8: (2, 7, 3), 9: (2, 6, 2),
             10: (1, 7, 1), 11: (5, 4, 5), 12: (2, 6, 4), 13: (4, 5, 6), 14: (3, 5, 4), 15: (3, 7, 3), 16: (2, 6, 2), 17: (2, 6, 2),
             18: (2, 7, 5), 19: (5, 3, 5), 20: (2, 7, 3), 21: (5, 3, 6), 22: (6, 3, 6), 23: (3, 5, 3), 24: (6, 4, 6), 25: (5, 4, 6),
             26: (2, 6, 2), 27: (6, 2, 5), 28: (6, 3, 5), 29: (1, 7, 1), 30: (1, 6, 1)}
FIELDS_P01 = {f'B_{m}{i:02d}': ITEMS_P01[i][k] for k, m in enumerate('WRB') for i in range(1, 31)}

def make_example_raw(path):
    base = dict(ResponseId='R_P01', participant_id='P01', project_path='B', protocol_version=PROTOCOL, phase='profile',
                Status=0, Finished=1, Progress=100, **{'Duration (in seconds)': 611}, zgoda=1)
    write_qualtrics_csv(path, [{**base, 'ResponseId': 'R_prev01', 'Status': 1, **FIELDS_P01}, {**base, **FIELDS_P01}], {})

In [ ]:
RAW_PATH = 'qualtrics_raw.csv'
if TRYB == 'przyklad':
    make_example_raw(RAW_PATH)
elif not os.path.exists(RAW_PATH):
    if not IN_COLAB:
        raise FileNotFoundError(f'Umieść eksport jako {os.path.join(WORKDIR, RAW_PATH)}')
    from google.colab import files
    up = files.upload()
    open(RAW_PATH, 'wb').write(next(iter(up.values())))
print('Plik źródłowy:', os.path.join(WORKDIR, RAW_PATH))

In [ ]:
def load_raw(path):
    """Eksport Qualtricsa: trzy wiersze nagłówkowe, dane od czwartego."""
    return pd.read_csv(path, skiprows=[1, 2])

def select_response(raw, participant_id, protocol_version, phase='profile'):
    """Dokładnie jedno wypełnienie: Status 0 (nie Preview 1, nie Test 2), Finished 1, zgoda 1, kod, wersja, faza."""
    # TODO: przefiltruj raw według sześciu warunków z opisu, sprawdź, że został dokładnie jeden wiersz, zwróć go
    raise NotImplementedError

raw = load_raw(RAW_PATH)
resp = select_response(raw, PARTICIPANT, PROTOCOL)
print('Wybrano wypełnienie', resp['ResponseId'])

## 3 · Długi plik 30 pozycji i średnie domen (rozdz. 6–7)

In [ ]:
def to_long(row):
    """30 wierszy: item_id, domain, W, R, B; braki jako NaN (nie 4 i nie 0); odpowiedź spoza 1–7 też jako brak."""
    # TODO: dla każdej pozycji 1–30 odczytaj W, R, B i dopisz domenę z KEY
    raise NotImplementedError

def score_domains(long):
    """Jeden wiersz na domenę: domain, n_w, w_sum, w_mean, n_r, r_mean, n_b, b_mean, status."""
    # TODO: dla każdej domeny liczba ważnych W/R/B, suma i średnie (tylko przy komplecie 6) oraz status
    raise NotImplementedError

_d = score_domains(to_long(FIELDS_P01)).set_index('domain')
check('dane P01: rekreacja W = 29/6, R = 4, B = 5,5', _d.loc['recreational', 'w_sum'] == 29 and close(_d.loc['recreational', 'w_mean'], 29 / 6)
      and close(_d.loc['recreational', 'r_mean'], 4) and close(_d.loc['recreational', 'b_mean'], 5.5))
check('dane P01: średnie W 1,333 / 2,333 / 2,333 / 4,833 / 5,833', all(close(_d.loc[d, 'w_mean'], v, 1e-3) for d, v in zip(KEY, [1.333, 2.333, 2.333, 4.833, 5.833])))
check('brak jednej pozycji → domena incomplete, bez średniej', score_domains(to_long({**FIELDS_P01, 'B_W24': ''})).set_index('domain').loc['recreational', 'status'] == 'incomplete')

long = to_long(resp)
long.to_csv('responses_clean.csv', index=False)
dom = score_domains(long).set_index('domain')
check_example('uczestnik = P01 z przewodnika (suma W rekreacji 29)', dom.loc['recreational', 'w_sum'] == 29, dom.loc['recreational', 'w_sum'])
dom[['n_w', 'w_sum', 'w_mean', 'n_r', 'r_mean', 'n_b', 'b_mean', 'status']].rename(index=DOMAIN_PL)

## 4 · Od W do kary za ryzyko i reguła (rozdz. 8)
$t = \frac{W - 1}{6}$, mapowanie główne $c = 1 - t$, wariant wrażliwości $c' = 1 - \sqrt{t}$. Reguła: Score = Benefit − c · Risk, gdzie Benefit i Risk ∈ [0, 1] to **założenia bodźca**, a nie pomiar osoby. Wynik jest indeksem reguły, a nie przewidywaną odpowiedzią W.

In [ ]:
def penalty(w_mean, mapping='linear'):
    # TODO: t, potem c dla 'linear' albo c' dla 'sqrt'
    raise NotImplementedError

def score_option(benefit, risk, c):
    return benefit - c * risk

def predict_pair(left, right, c, eps=TIE_EPS):
    """left/right: (benefit, risk). Zwraca LEFT, RIGHT, TIE albo NO_PREDICTION (c = None)."""
    # TODO: brak c → NO_PREDICTION; różnica Score prawej i lewej; próg eps
    raise NotImplementedError

def app_penalty(domains, domain=APP_DOMAIN, mapping='linear'):
    """c do personalizacji: tylko kompletna domena, dla której istnieje pula (rekreacja). Gotowe."""
    if domain != 'recreational' or domains.loc[domain, 'status'] != 'complete':
        return None
    return penalty(domains.loc[domain, 'w_mean'], mapping)

def explain_pair(left, right, c):
    """Wyjaśnienie pokazywane po serii. Gotowe."""
    sl, sr = score_option(*left, c), score_option(*right, c)
    return (f'Reguła porównała atrakcyjność i niepewność obu planów. Przy Twoim profilu rekreacyjnym niepewność obniża ocenę '
            f'z wagą c = {c:.2f} (0 = wcale, 1 = w pełni). Wynik lewej opcji: {sl:.3f}, prawej: {sr:.3f}.')

check('W = 29/6 → c = 13/36', close(penalty(29 / 6), 13 / 36))
check("W = 29/6 → c' = 0,200694", close(penalty(29 / 6, 'sqrt'), 0.200694, 1e-6))
P1, P2 = ((0.65, 0.20), (0.90, 0.60)), ((0.60, 0.20), (0.72, 0.60))
check('P1 przy c P01: 0,577778 vs 0,683333 → RIGHT', close(score_option(*P1[0], C_P01), 0.577778, 1e-6) and predict_pair(*P1, C_P01) == 'RIGHT')
check("P2: LEFT przy c, RIGHT przy c'", predict_pair(*P2, C_P01) == 'LEFT' and predict_pair(*P2, penalty(29 / 6, 'sqrt')) == 'RIGHT')
check('baseline c0 = 0,8: P1 → LEFT', predict_pair(*P1, C0) == 'LEFT')
check('inna domena → brak personalizacji', app_penalty(_d, 'financial') is None)

c = app_penalty(dom); c_alt = app_penalty(dom, mapping='sqrt')
check_example('c P01 = 13/36', c is not None and close(c, 13 / 36), c)
profil = dom.reset_index()
profil['app_domain'] = APP_DOMAIN; profil['mapping'] = 'linear'; profil['c'] = c; profil['baseline_c'] = C0
profil['model_version'] = MODEL_VERSION; profil.insert(0, 'participant_id', PARTICIPANT)
profil.to_csv('profil.csv', index=False)

## 5 · Gotowy wariant: pula 16 scenariuszy i stała reguła wyboru 12 (rozdz. 9)
Próg $c^* = \frac{\Delta \text{Benefit}}{\Delta \text{Risk}}$ to wartość c, przy której obie opcje mają ten sam wynik. Przy c < c* reguła wybiera prawą opcję, przy c > c* lewą. Para różnicuje modele, gdy c* leży między c a c0.

Reguła wyboru:
1. Odrzuć pary z |c* − c| < 0,02 albo |c* − c0| < 0,02.
2. Weź najwyżej 6 par różnicujących w kolejności numerów.
3. Uzupełnij parami zgodnymi.
4. Jeśli par zgodnych brakuje, dobierz kolejne pary różnicujące.

In [ ]:
POOL = [('P01', 'Znana gra terenowa w przygotowanym parku', .65, .20, 'Nowa gra orientacyjna na nieznanym, przygotowanym terenie', .90, .60),
        ('P02', 'Spacer znaną trasą w lesie miejskim', .60, .20, 'Wycieczka rowerowa nową trasą według mapy', .72, .60),
        ('P03', 'Seans w kinie, znany gatunek filmu', .55, .10, 'Pokaz filmu-niespodzianki bez zapowiedzi tytułu', .75, .60),
        ('P04', 'Znana gra planszowa ze znajomymi', .55, .10, 'Nowa gra planszowa o rozbudowanych zasadach', .61, .50),
        ('P05', 'Stała kawiarnia blisko domu', .50, .10, 'Nowa kawiarnia w innej dzielnicy, dojazd z przesiadką', .58, .50),
        ('P06', 'Kręgle w znanym klubie', .55, .15, 'Pierwszy pokój zagadek (escape room) w nowym miejscu', .65, .55),
        ('P07', 'Piknik w pobliskim parku', .60, .15, 'Piknik nad rzeką za miastem, dojazd pociągiem regionalnym', .74, .55),
        ('P08', 'Basen kryty', .50, .10, 'Wypożyczenie kajaka na spokojnym jeziorze przy niepewnej pogodzie', .68, .50),
        ('P09', 'Wizyta w znanym muzeum', .50, .15, 'Zwiedzanie miasta z nową grą miejską w aplikacji', .75, .65),
        ('P10', 'Koncert znanego zespołu w klubie', .65, .20, 'Koncert plenerowy nieznanych wykonawców', .87, .60),
        ('P11', 'Warsztat ceramiki u znanego prowadzącego', .55, .20, 'Warsztat nowej techniki u nowego prowadzącego', .85, .70),
        ('P12', 'Rower po znanym bulwarze', .55, .15, 'Rower nowym szlakiem poza miastem', .81, .55),
        ('P13', 'Gotowanie znanego przepisu', .50, .10, 'Gotowanie potrawy z nieznanej kuchni', .85, .60),
        ('P14', 'Wycieczka z przewodnikiem znaną trasą', .55, .15, 'Samodzielna wycieczka nową trasą', .85, .55),
        ('P15', 'Fotografowanie w znanym parku', .45, .10, 'Fotospacer o świcie w nowym miejscu, zależny od pogody i dojazdu', .85, .60),
        ('P16', 'Gra w piłkę ze znajomymi na znanym boisku', .55, .15, 'Gra z nowymi osobami umówiona przez aplikację', .91, .55)]
pool = pd.DataFrame(POOL, columns=['pair_id', 'left_desc', 'benefit_L', 'risk_L', 'right_desc', 'benefit_R', 'risk_R'])
pool.to_csv('pool.csv', index=False)

def classify_pool(pool, c, c0):
    """Dodaje c_star, pred_personal, pred_baseline, cls ∈ {'odrzucona', 'różnicująca', 'zgodna'}."""
    # TODO: c*, predykcje obu modeli, klasa pary według kroku 1 reguły i porównania predykcji
    raise NotImplementedError

def select_cases(cl, n_total=12, n_diff_max=6):
    # TODO: kroki 2–4 reguły; zwróć posortowaną listę pair_id
    raise NotImplementedError

_cl = classify_pool(pool, C_P01, C0)
check('c P01: odpadają P07 i P15; 9 różnicujących, 5 zgodnych', list(_cl.loc[_cl.cls == 'odrzucona', 'pair_id']) == ['P07', 'P15']
      and ((_cl.cls == 'różnicująca').sum(), (_cl.cls == 'zgodna').sum()) == (9, 5))
check('c P01: wybrane P01–P06, P08–P12, P16', select_cases(_cl) == ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P08', 'P09', 'P10', 'P11', 'P12', 'P16'])

cl = classify_pool(pool, c, C0)
selected = select_cases(cl)
print(f'Uczestnik {PARTICIPANT}: wybrane pary {", ".join(selected)} | różnicujące: {int(cl[cl.pair_id.isin(selected)].cls.eq("różnicująca").sum())}')

In [ ]:
cases = cl[cl.pair_id.isin(selected)].copy()
cases.insert(0, 'case_id', cases.pair_id)
cases['pred_alt_sqrt'] = [predict_pair((r.benefit_L, r.risk_L), (r.benefit_R, r.risk_R), c_alt) for r in cases.itertuples()]
cases['pred_c0_extra'] = [predict_pair((r.benefit_L, r.risk_L), (r.benefit_R, r.risk_R), C0_EXTRA) for r in cases.itertuples()]
rng = np.random.default_rng(20261020)
cases['display_order'] = rng.permutation(len(cases)) + 1
cases['left_text'] = '### ' + cases.left_desc            # uczestnik widzi pełne opisy, bez indeksów
cases['right_text'] = '### ' + cases.right_desc
cases['left_code'], cases['right_code'] = 'LEFT', 'RIGHT'
cases['explanation'] = [explain_pair((r.benefit_L, r.risk_L), (r.benefit_R, r.risk_R), c) if c is not None else '' for r in cases.itertuples()]
cases['model_version'] = MODEL_VERSION
cases['params'] = f'domain={APP_DOMAIN}|c={c:.6f}|c0={C0}|eps={TIE_EPS}' if c is not None else 'c=None'
PROFILE_OK = c is not None
PROFILE_NOTE = 'valid' if c is not None else 'domena rekreacyjna niekompletna albo inna domena niż pula'
SESSION_EXTRA = dict(title='Jak spędzisz wolne popołudnie?', left_button='Wybieram opcję po lewej', right_button='Wybieram opcję po prawej',
                     code_labels={'LEFT': 'opcja po lewej', 'RIGHT': 'opcja po prawej'},
                     intro='Wyobraź sobie wolne popołudnie. W każdej sytuacji wybierz jeden z dwóch sposobów jego spędzenia. '
                           'Sytuacje są hipotetyczne. Nie ma poprawnych odpowiedzi.')
cases.sort_values('display_order')[['display_order', 'case_id', 'pred_personal', 'pred_baseline', 'pred_alt_sqrt']]

## 6 · `model.py`, zamrożenie, kontrakt i aplikacja

In [ ]:
FUNCS = [to_long, score_domains, penalty, score_option, predict_pair, app_penalty, explain_pair]
CONSTS = ['KEY', 'APP_DOMAIN', 'C0', 'TIE_EPS', 'NEAR', 'MODEL_VERSION']
src = ['"""model.py – wygenerowane z notatnika ścieżki B; nie edytuj ręcznie."""', 'import math', 'import numpy as np', 'import pandas as pd', '']
src += [f'{n} = {globals()[n]!r}' for n in CONSTS] + [''] + [inspect.getsource(f) for f in FUNCS]
open('model.py', 'w', encoding='utf-8').write('\n'.join(src))
sys.path.insert(0, WORKDIR)
if 'model' in sys.modules: del sys.modules['model']
import model
check('model.py: P2 → LEFT przy c P01', model.predict_pair(*P2, C_P01) == 'LEFT')

## Zamrożenie przed sesją
Komórka zapisuje `cases.csv` z opisami opcji i predykcjami obu modeli oraz `session_config.json` ze skrótem SHA-256. Skrót wpisz do dziennika **przed sesją**. W trybie `wlasne` notatnik nie nadpisze zamrożonych przypadków: jeśli po zamrożeniu zmienisz dane albo kod, komórka zatrzyma się z błędem. 

In [ ]:
new_csv = cases.to_csv(index=False).encode()
new_hash = hashlib.sha256(new_csv).hexdigest()
if not PROFILE_OK:
    raise RuntimeError(f'Profil nie pozwala na personalizację ({PROFILE_NOTE}). Nie zamrażamy przypadków i nie prowadzimy sesji.')
if TRYB == 'wlasne' and os.path.exists('session_config.json'):
    frozen = json.load(open('session_config.json', encoding='utf-8'))
    if frozen['frozen_sha256'] != new_hash:
        raise RuntimeError('Przypadki są już zamrożone, a nowe obliczenia dają inny cases.csv. Nie nadpisuję. '
                           'Jeśli zmiana nastąpiła PRZED sesją, usuń świadomie session_config.json i opisz to w dzienniku; '
                           'po sesji wymaga to nowej wersji modelu i nowego zestawu.')
    print('Zamrożenie bez zmian:', new_hash)
else:
    open('cases.csv', 'wb').write(new_csv)
    json.dump(dict(contract='B_2006_v1', participant_id=PARTICIPANT, session_id=SESSION_ID, frozen_sha256=new_hash,
                   frozen_at=now_iso(), model_version=MODEL_VERSION, seed=20261020, instructor_pin=INSTRUCTOR_PIN,
                   profile_ok=PROFILE_OK, profile_note=PROFILE_NOTE, **SESSION_EXTRA),
              open('session_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('Zamrożono cases.csv | SHA-256:', new_hash)
FROZEN_HASH = file_sha256('cases.csv')

In [ ]:
cfg = json.load(open('session_config.json', encoding='utf-8'))
_cases = pd.read_csv('cases.csv')
missing = [c for c in ['case_id', 'display_order', 'left_text', 'right_text', 'left_code', 'right_code', 'pred_personal', 'pred_baseline', 'model_version', 'params', 'explanation'] if c not in _cases.columns]
check('cases.csv: wymagane kolumny', not missing or print('Brakuje:', missing))
check('session_config.json: kontrakt i zgodny skrót cases.csv', cfg['contract'] == 'B_2006_v1' and cfg['frozen_sha256'] == file_sha256('cases.csv'))
check('model.py: wymagane funkcje', all(hasattr(model, n) for n in ['predict_pair', 'explain_pair']))
check('profil.csv istnieje', os.path.exists('profil.csv'))

## Aplikacja i sesja z uczestnikiem
Komórka z kodem aplikacji jest zwinięta. **Nie edytuj jej.** Zapisuje plik `app.py` w folderze projektu.

**Przebieg sesji:**
1. Uczestnik widzi pary scenariuszy z pełnymi opisami, bez indeksów Benefit/Risk, i wybiera bez podpowiedzi. Każdy wybór trafia od razu do `validation.csv`.
2. Po wszystkich wyborach aplikacja pokazuje rekomendacje i wyjaśnienia, a potem zbiera jedną ocenę dopasowania i jedną ocenę użyteczności serii (1–7).

Widok prowadzącego (panel boczny, kod `INSTRUCTOR_PIN`) pokazuje kontrakt, zamrożone przypadki i log. Nie otwieraj go przy uczestniku. Odświeżenie strony nie gubi danych: aplikacja wznawia sesję od pierwszego niezapisanego kroku.

In [ ]:
#@title Kod aplikacji – gotowy, nie edytuj { display-mode: "form" }
APP_SOURCE = r'''"""Sesja walidacyjna – ścieżki A, B i C (wybór jednej z dwóch opcji).

Gotowa aplikacja kursu: zespół jej NIE edytuje. Aplikacja niczego nie przewiduje sama:
opisy opcji, predykcje i wyjaśnienia pochodzą z zamrożonego cases.csv utworzonego w notatniku.

Uruchomienie:  streamlit run app.py -- --data <folder z plikami projektu>
Folder musi zawierać: cases.csv, profil.csv, model.py, session_config.json (tworzy je notatnik).
Aplikacja zapisuje: validation.csv (po każdym wyborze) i session_<id>.json (oceny serii).
"""
import argparse
import hashlib
import json
import os
import sys
from datetime import datetime, timezone

import pandas as pd
import streamlit as st

APP_VERSION = 'pred_app_v1'
REQUIRED_CASES = ['case_id', 'display_order', 'left_text', 'right_text', 'left_code', 'right_code',
                  'pred_personal', 'pred_baseline', 'model_version', 'params', 'explanation']
REQUIRED_CFG = ['contract', 'participant_id', 'session_id', 'frozen_sha256', 'instructor_pin', 'profile_ok',
                'title', 'intro', 'left_button', 'right_button', 'code_labels']


def data_dir():
    p = argparse.ArgumentParser()
    p.add_argument('--data', default='.')
    return os.path.abspath(p.parse_known_args(sys.argv[1:])[0].data)


DATA = data_dir()
path = lambda name: os.path.join(DATA, name)
now_iso = lambda: datetime.now(timezone.utc).isoformat(timespec='seconds')
sha256 = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()


def load_project():
    """Wczytuje pliki i sprawdza kontrakt. Zwraca (cfg, cases, błędy, ostrzeżenia)."""
    errors, warnings = [], []
    for name in ['cases.csv', 'profil.csv', 'model.py', 'session_config.json']:
        if not os.path.exists(path(name)):
            errors.append(f'Brak pliku {name} w folderze {DATA}. Uruchom notatnik do sekcji „Zamrożenie”.')
    if errors:
        return {}, None, errors, warnings
    cfg = json.load(open(path('session_config.json'), encoding='utf-8'))
    missing = [k for k in REQUIRED_CFG if k not in cfg]
    if missing:
        errors.append(f'session_config.json: brak pól {", ".join(missing)}')
    cases = pd.read_csv(path('cases.csv'), dtype={'case_id': str})
    missing = [c for c in REQUIRED_CASES if c not in cases.columns]
    if missing:
        errors.append(f'cases.csv: brak kolumn {", ".join(missing)}')
    if sha256(path('cases.csv')) != cfg.get('frozen_sha256'):
        errors.append('cases.csv różni się od wersji zamrożonej (SHA-256 nie zgadza się z session_config.json).')
    if not cfg.get('profile_ok', False):
        errors.append(f'Profil nie pozwala na personalizację: {cfg.get("profile_note", "")}')
    if not errors:
        bad = set(cases.pred_personal.astype(str)) - set(cfg['code_labels']) - {'TIE', 'NO_PREDICTION'}
        if bad:
            errors.append(f'cases.csv: nieznane kody predykcji {bad}')
        cases = cases.sort_values('display_order').reset_index(drop=True)
    return cfg, cases, errors, warnings


def read_log():
    if os.path.exists(path('validation.csv')):
        return pd.read_csv(path('validation.csv'), dtype={'session_id': str, 'case_id': str})
    return pd.DataFrame(columns=['session_id', 'case_id'])


def append_log(row):
    log = pd.concat([read_log(), pd.DataFrame([row])], ignore_index=True)
    tmp = path('validation.csv.tmp')
    log.to_csv(tmp, index=False)
    os.replace(tmp, path('validation.csv'))       # zapis atomowy


st.set_page_config(page_title='Sesja walidacyjna', page_icon='🧭', layout='wide')
cfg, cases, errors, warnings = load_project()
SESSION = str(cfg.get('session_id', 'S01'))
SESSION_JSON = path(f'session_{SESSION}.json')

with st.sidebar:
    st.caption(f'{APP_VERSION} · kontrakt {cfg.get("contract", "?")}')
    pin = st.text_input('Kod prowadzącego', type='password')
    instructor = bool(cfg) and pin != '' and pin == str(cfg.get('instructor_pin'))

if instructor:
    st.title('Widok prowadzącego')
    for e in errors:
        st.error(e)
    for w in warnings:
        st.warning(w)
    if not errors:
        st.success(f'Kontrakt spełniony · cases.csv zgodny z zamrożeniem · SHA-256 {cfg["frozen_sha256"][:16]}…')
        st.subheader('Profil')
        st.dataframe(pd.read_csv(path('profil.csv')), hide_index=True)
        st.subheader('Zamrożone przypadki')
        st.dataframe(cases[['display_order', 'case_id', 'left_text', 'right_text', 'pred_personal', 'pred_baseline']], hide_index=True)
    log = read_log()
    n = int((log.session_id.astype(str) == SESSION).sum()) if len(log) else 0
    st.subheader(f'Log sesji {SESSION}: {n} zapisanych wyborów')
    st.dataframe(log, hide_index=True)
    if os.path.exists(path('validation.csv')):
        st.download_button('Pobierz validation.csv', open(path('validation.csv'), 'rb').read(), 'validation.csv', 'text/csv')
    if os.path.exists(SESSION_JSON):
        st.json(json.load(open(SESSION_JSON, encoding='utf-8')))
    st.stop()

st.title(cfg.get('title', 'Sesja walidacyjna'))
if errors:
    st.error('Aplikacja nie jest gotowa do sesji. Poproś prowadzącego o sprawdzenie (widok prowadzącego w panelu bocznym).')
    st.stop()

log = read_log()
done = log[log.session_id.astype(str) == SESSION]
remaining = cases[~cases.case_id.isin(set(done.case_id.astype(str)))]
if os.path.exists(SESSION_JSON):
    st.success('Sesja zakończona. Dziękujemy za udział!')
    st.stop()

if 'started' not in st.session_state:
    st.session_state.started = len(done) > 0
if not st.session_state.started:
    st.write(cfg['intro'])
    st.write(f'Zobaczysz {len(cases)} sytuacji. W każdej wybierz jedną z dwóch możliwości. '
             'Wyjaśnienia działania systemu zobaczysz dopiero po zapisaniu wszystkich wyborów.')
    if st.button('Rozpocznij', type='primary'):
        st.session_state.started = True
        st.session_state.started_at = now_iso()
        st.rerun()
    st.stop()

if len(remaining):
    row = remaining.iloc[0]
    k = len(done) + 1
    st.progress((k - 1) / len(cases), text=f'Sytuacja {k} z {len(cases)}')
    st.caption(cfg['intro'])
    cl, cr = st.columns(2)
    for col, txt in ((cl, row.left_text), (cr, row.right_text)):
        with col:
            with st.container(border=True):
                st.markdown(txt)

    def choose(position):
        code = row.left_code if position == 'left' else row.right_code
        rec = dict(participant_id=cfg['participant_id'], session_id=SESSION, case_id=str(row.case_id),
                   display_order=int(row.display_order), chosen_position=position, answer=code, answered_at=now_iso(),
                   model_version=row.model_version, params=row.params, app_version=APP_VERSION)
        rec.update({c: row[c] for c in cases.columns if c.startswith('pred_')})
        append_log(rec)

    b1, b2 = st.columns(2)
    b1.button(cfg['left_button'], key=f'L_{row.case_id}', on_click=choose, args=('left',), use_container_width=True)
    b2.button(cfg['right_button'], key=f'R_{row.case_id}', on_click=choose, args=('right',), use_container_width=True)
    st.stop()

st.progress(1.0, text='Wszystkie wybory zapisane')
st.header('Jak system opisuje Twoje wybory')
st.write('Teraz, po zapisaniu wszystkich decyzji, pokazujemy rekomendacje systemu i ich uzasadnienie. '
         'Rekomendacja opisuje działanie modelu, a nie poprawność Twojej decyzji.')
ans = dict(zip(done.case_id.astype(str), done.answer))
labels = cfg['code_labels']
for _, row in cases.iterrows():
    mine = ans.get(str(row.case_id))
    with st.expander(f'Sytuacja {int(row.display_order)}: wybrano „{labels.get(mine, mine)}”'):
        c1, c2 = st.columns(2)
        c1.markdown(row.left_text); c2.markdown(row.right_text)
        rec = row.pred_personal
        if rec in labels:
            st.write(f'**System rekomendowałby: {labels[rec]}.** {row.explanation}')
        else:
            st.write(f'System nie rozstrzyga tej sytuacji ({rec}). {row.explanation}')

st.header('Ocena całej serii')
fit = st.slider('Na ile rekomendacje były dopasowane do Twoich wyborów? (1 = wcale, 7 = bardzo)', 1, 7, 4)
useful = st.slider('Na ile wyjaśnienia były dla Ciebie użyteczne? (1 = wcale, 7 = bardzo)', 1, 7, 4)
if st.button('Zakończ sesję', type='primary'):
    json.dump(dict(participant_id=cfg['participant_id'], session_id=SESSION, rating_fit=int(fit), rating_usefulness=int(useful),
                   started_at=st.session_state.get('started_at', ''), finished_at=now_iso(), n_choices=len(done),
                   cases_sha256=cfg['frozen_sha256'], app_version=APP_VERSION),
              open(SESSION_JSON, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    st.rerun()
'''
open('app.py', 'w', encoding='utf-8').write(APP_SOURCE)
print('Zapisano app.py w', WORKDIR)

In [ ]:
#@title ▶ Uruchom aplikację w Colabie { display-mode: "form" }
URUCHOM = False  #@param {type:"boolean"}
import subprocess, socket, time
def _port_open(port):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0
if URUCHOM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'], check=True)
    if not _port_open(8501):
        subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', os.path.join(WORKDIR, 'app.py'),
                          '--server.port', '8501', '--server.headless', 'true',
                          '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
                          '--', '--data', WORKDIR],
                         stdout=open(os.path.join(WORKDIR, 'streamlit.log'), 'w'), stderr=subprocess.STDOUT)
        for _ in range(60):
            if _port_open(8501): break
            time.sleep(1)
    if IN_COLAB:
        from google.colab import output
        output.serve_kernel_port_as_window(8501, anchor_text='▶ Otwórz aplikację w nowej karcie')
    else:
        print('Aplikacja: http://localhost:8501')
else:
    print('Zaznacz URUCHOM i uruchom komórkę ponownie. Jeśli link nie działa, przeprowadź sesję lokalnie (komórka niżej).')

**Sesja lokalna (zalecana przy uczestniku).** Colab może się rozłączyć w trakcie sesji. Pewniej jest uruchomić aplikację na laptopie:
1. Uruchom komórkę poniżej i pobierz `pakiet_sesji_B.zip` (`app.py`, `model.py`, `profil.csv`, `cases.csv`, `session_config.json`).
2. Na laptopie: `pip install streamlit pandas`, potem w rozpakowanym folderze `streamlit run app.py -- --data .`
3. Po sesji skopiuj `validation.csv` i `session_S01.json` do folderu projektu na Dysku i uruchom analizę w trybie `wlasne`.

In [ ]:
import zipfile
with zipfile.ZipFile('pakiet_sesji_B.zip', 'w') as z:
    for f in ['app.py', 'model.py', 'profil.csv', 'cases.csv', 'session_config.json']:
        z.write(f)
print('Zapisano', os.path.join(WORKDIR, 'pakiet_sesji_B.zip'))
if IN_COLAB:
    from google.colab import files
    files.download('pakiet_sesji_B.zip')

## 7 · Analiza walidacji i wrażliwości (rozdz. 11)
- `przyklad`: log w formacie aplikacji z **syntetycznymi** odpowiedziami.
- `wlasne`: `validation.csv` i `session_S01.json` zapisane przez aplikację.

In [ ]:
UNAMBIGUOUS = lambda s: s.notna() & ~s.isin(['TIE', 'NO_PREDICTION'])

def summarize(df, pred='pred_personal', base='pred_baseline', ans='answer'):
    """Podsumowanie walidacji wg przewodnika.

    - pokrycie: liczba prób z jednoznaczną predykcją (bez TIE i NO_PREDICTION),
    - trafność obu modeli na WSPÓLNYM zbiorze prób z odpowiedzią i jednoznaczną predykcją obu,
    - podział 2×2: oba trafiły / tylko indywidualny / tylko bazowy / żaden,
    - liczba prób wspólnego zbioru, w których modele się różnią.
    """
    # TODO: 1) wybierz wspólny zbiór: odpowiedź jest, obie predykcje jednoznaczne (UNAMBIGUOUS),
    #       2) policz trafienia obu modeli na tym zbiorze i podział 2×2,
    #       3) zwróć słownik o kluczach: n_trials, n_answered, coverage_personal,
    #          coverage_baseline, n_common, hits_personal, hits_baseline, both,
    #          only_personal, only_baseline, none, n_differ.
    raise NotImplementedError

def report(s, name_p='model indywidualny', name_b='model bazowy'):
    pct = lambda k, m: f'{k}/{m} = {100*k/m:.0f}%' if m else f'{k}/{m}'
    print(f"Pokrycie: {name_p} {s['coverage_personal']}/{s['n_trials']}, {name_b} {s['coverage_baseline']}/{s['n_trials']}")
    print(f"Trafność na wspólnym zbiorze: {name_p} {pct(s['hits_personal'], s['n_common'])}, "
          f"{name_b} {pct(s['hits_baseline'], s['n_common'])}")
    print(f"2×2: oba {s['both']}, tylko indywidualny {s['only_personal']}, tylko bazowy {s['only_baseline']}, żaden {s['none']}")
    print(f"Próby wspólnego zbioru, w których modele się różnią: {s['n_differ']}/{s['n_common']}")
    print('TIE i NO_PREDICTION nie są ani trafieniem, ani pomyłką. Wynik opisuje test jednej osoby.')

In [ ]:
_t = pd.DataFrame({'pred_personal': ['A', 'B', 'TIE', 'A', 'A'],
                   'pred_baseline': ['A', 'A', 'B', 'NO_PREDICTION', 'B'],
                   'answer':        ['A', 'B', 'B', 'A', None]})
_s = summarize(_t)
check('pokrycie liczy tylko jednoznaczne predykcje', (_s['coverage_personal'], _s['coverage_baseline']) == (4, 4))
check('wspólny zbiór pomija TIE, NO_PREDICTION i brak odpowiedzi', _s['n_common'] == 2)
check('podział 2×2', (_s['both'], _s['only_personal'], _s['only_baseline'], _s['none']) == (1, 1, 0, 0))

In [ ]:
def synthetic_session():
    ans = {'P01': 'RIGHT', 'P02': 'LEFT', 'P03': 'RIGHT', 'P04': 'LEFT', 'P05': 'LEFT', 'P06': 'RIGHT', 'P07': 'LEFT', 'P08': 'LEFT',
           'P09': 'RIGHT', 'P10': 'RIGHT', 'P11': 'RIGHT', 'P12': 'RIGHT', 'P13': 'RIGHT', 'P14': 'LEFT', 'P15': 'RIGHT', 'P16': 'RIGHT'}
    v = pd.read_csv('cases.csv', dtype={'case_id': str}).sort_values('display_order')
    log = v[['case_id', 'display_order', 'model_version', 'params'] + [c for c in v.columns if c.startswith('pred_')]].copy()
    log.insert(0, 'participant_id', PARTICIPANT); log.insert(1, 'session_id', SESSION_ID)
    log['answer'] = log.case_id.map(ans)
    log['chosen_position'] = log.answer.str.lower()
    log['answered_at'] = now_iso(); log['app_version'] = 'synthetic'
    return log, dict(rating_fit=5, rating_usefulness=5, n_choices=len(log))

In [ ]:
check('cases.csv nie zmienił się od zamrożenia', file_sha256('cases.csv') == FROZEN_HASH)
validation, session = None, None
if TRYB == 'przyklad':
    validation, session = synthetic_session()          # odpowiedzi syntetyczne w formacie logu aplikacji
elif os.path.exists('validation.csv'):
    validation = pd.read_csv('validation.csv', dtype={'session_id': str, 'case_id': str})
    validation = validation[validation.session_id == SESSION_ID]
    if os.path.exists(f'session_{SESSION_ID}.json'):
        session = json.load(open(f'session_{SESSION_ID}.json', encoding='utf-8'))
    _frozen = pd.read_csv('cases.csv', dtype={'case_id': str})
    check('log zawiera wyłącznie zamrożone przypadki', set(validation.case_id) <= set(_frozen.case_id.astype(str)))
    if 'pred_personal' in validation:
        _m = validation.merge(_frozen, on='case_id', suffixes=('', '_f'))
        check('predykcje w logu zgodne z zamrożonymi', ((_m.pred_personal == _m.pred_personal_f) & (_m.pred_baseline == _m.pred_baseline_f)).all())
    print(f'Wczytano {len(validation)} zapisów z sesji {SESSION_ID}.', '' if session else '⚠ Brak pliku session_*.json.')
else:
    print('Brak validation.csv: najpierw przeprowadź sesję.')

In [ ]:
if validation is not None:
    s = summarize(validation)
    report(s, 'reguła z c z profilu', 'baseline c0 = 0,8')
    check_example('trafność 10/12 wobec 5/12', (s['hits_personal'], s['hits_baseline'], s['n_common']) == (10, 5, 12))
    # TODO: analiza wrażliwości: zmiany predykcji przy c' oraz trafność wariantu c' i baseline c0 = 0,5

In [ ]:
if session:
    print('Ocena dopasowania serii (1–7):', session.get('rating_fit'), '| ocena użyteczności serii (1–7):', session.get('rating_usefulness'))

## 8 · Pytania do raportu
1. Dlaczego Score = 0,68 nie można porównać ze średnią W = 4,83?
2. Co mówi wynik 10/12 wobec 5/12 przy baseline, który prawie zawsze wybiera lewą opcję?
3. Co trzeba zrobić, jeśli zespół chce zastosować profil domeny finansowej?